# GoProceed prospecting — Wave 6 validation

Reproducible companion analysis for the unified lead base as of **2026-08-24** (Europe/Kyiv). The objective is to expand the pool of construction-project prospects while preserving the previously reviewed 51 leads and avoiding false claims of product demand.

## tl;dr

Wave 6 expands the reviewed base from 1,688 to 1,929 companies. The new search covers additional project-heavy verticals such as hydrotechnical works, wells, industrial tanks, traffic systems, energy infrastructure, demolition, waste facilities and specialized coatings. Tender awards are used only as workflow/activity signals—not as evidence that a company wants GoProceed.

## Context & Methods

Sources: the existing v6 prospect file, the official Prozorro API-derived Wave 6 supplier set, the normalization summary, browser-verification log and the v7 validation output. Records were filtered to awarded construction/installation work, then deduplicated primarily by EDRPOU. Design-only, expertise, supervision, inventory, survey, research, rental and training records were excluded.

### Key Assumptions

- A Prozorro award confirms legal/project activity, not product interest.
- A future published completion date is a planning signal, not proof of an active site.
- Current product coverage is limited to the seeded N.14/N.15 requirements.
- Other verticals receive a two-week workflow-discovery offer, not a coverage claim.
- Attributed award value is a prioritization signal and is not TAM or revenue.

## Data

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

OUT = Path('/Users/akisliy/Downloads/GoProceed/outputs/01a033d9-c008-7011-bf7b-e1dbd14e2e9d')
def load(name):
    return json.loads((OUT / name).read_text(encoding='utf-8'))

base_v6 = load('prospects_unified_v6_2026-08-24.json')
wave6 = load('prozorro_wave6_supplier_leads_final_2026-08-24.json')
wave6_summary = load('prozorro_wave6_summary_final_2026-08-24.json')
normalization = load('wave6_normalization_summary_2026-08-24.json')
v7 = load('prospects_unified_v7_2026-08-24.json')
validation = load('wave6_v7_validation_2026-08-24.json')
browser_checks = load('wave6_browser_verification_2026-08-24.json')
df = pd.DataFrame(v7)
wave6_df = df[df['lead_id'].astype(str).str.startswith('W6-')].copy()
print({'v6_rows': len(base_v6), 'wave6_supplier_candidates': len(wave6), 'v7_rows': len(df), 'browser_spot_checks': len(browser_checks['verified'])})

{'v6_rows': 1688, 'wave6_supplier_candidates': 288, 'v7_rows': 1929, 'browser_spot_checks': 6}


## Results

In [2]:
lane_counts = df['lane'].value_counts().to_dict()
intent_counts = df['intent_priority'].fillna('—').replace('', '—').value_counts().to_dict()
headline = {
    'total_leads': len(df),
    'unique_new_wave6': normalization['unique_new_supplier_leads'],
    'wave6_overlaps_merged': normalization['overlaps_with_base_1688'],
    'tender_signal_accounts': int((df['tender_count'].fillna(0) > 0).sum()),
    'future_completion_signal_accounts': int((df['live_project_count'].fillna(0) > 0).sum()),
    'high_confidence_accounts': int((df['evidence_confidence'] == 'high').sum()),
    'pilot_now': lane_counts.get('Pilot now', 0),
    'expansion_discovery': lane_counts.get('Expansion discovery', 0),
    'intent_I1': intent_counts.get('I1', 0),
    'intent_I2': intent_counts.get('I2', 0),
}
display(pd.DataFrame([headline]).T.rename(columns={0: 'value'}))

,value
total_leads,1929
unique_new_wave6,241
wave6_overlaps_merged,44
tender_signal_accounts,1670
future_completion_signal_accounts,931
high_confidence_accounts,1863
pilot_now,512
expansion_discovery,1375
intent_I1,110
intent_I2,318


In [3]:
assert validation['ok'] is True
assert len(df) == 1929
assert df['lead_id'].nunique() == len(df)
edrpou = df['edrpou'].fillna('').astype(str).str.replace(r'\D', '', regex=True)
assert edrpou[edrpou.ne('')].nunique() == edrpou.ne('').sum()
previous = df[df['wave'] == 'Перевірено 51 — 24.08']
assert len(previous) == 51
assert (previous['evidence_confidence'] == 'high').all()
assert lane_counts == {'Expansion discovery': 1375, 'Pilot now': 512, 'Requalify': 34, 'Later / requirements': 8}
print('Validation passed: unique IDs/EDRPOU, 51/51 prior leads preserved, lane totals reconcile.')

Validation passed: unique IDs/EDRPOU, 51/51 prior leads preserved, lane totals reconcile.


In [4]:
top_cols = ['lead_id','company_name','edrpou','segment','lane','intent_score','intent_priority','tender_count','live_project_count','domain']
display(wave6_df.sort_values(['intent_score','total_award_value_uah'], ascending=False)[top_cols].head(15).reset_index(drop=True))

,lead_id,company_name,edrpou,segment,lane,intent_score,intent_priority,tender_count,live_project_count,domain
0,W6-001,"ПРИВАТНЕ ПІДПРИЄМСТВО ""УКРНАДРА-ПІВДЕНЬ""",34994941,Свердловини / водозабір,Expansion discovery,93,I1,12,2,ukrnadra.com
1,W6-002,"ТОВАРИСТВО З ОБМЕЖЕНОЮ ВІДПОВІДАЛЬНІСТЮ ""АРІС-...",40538615,Полігони відходів,Expansion discovery,91,I1,3,3,
2,W6-003,"ТОВАРИСТВО З ОБМЕЖЕНОЮ ВІДПОВІДАЛЬНІСТЮ ""Будів...",45655160,Резервуари / промислові ємності,Expansion discovery,88,I1,3,1,
3,W6-004,"Товариство з обмеженою відповідальністю ""Анкор""",32290523,Днопоглиблення / порти,Expansion discovery,87,I1,4,2,ankor-group.com
4,W6-005,"ТОВАРИСТВО З ОБМЕЖЕНОЮ ВІДПОВІДАЛЬНІСТЮ ""СПЕЦМ...",37528539,Резервуари / промислові ємності,Expansion discovery,87,I1,9,3,specmontazinzhiniring.com.ua
5,W6-006,"ТОВАРИСТВО З ОБМЕЖЕНОЮ ВІДПОВІДАЛЬНІСТЮ ""КРАФТ...",45747041,Світлофори / дорожня автоматика,Pilot now,87,I1,5,4,
6,W6-007,"ТзОВ ""ЛЕВОКСС""",41432667,Гідротехнічні споруди,Expansion discovery,84,I1,3,3,
7,W6-008,ТОВ «ГОЛОВНИЙ ПОЖЕЖНИЙ ОФІС»,45147564,Пожежні системи,Expansion discovery,84,I1,4,2,
8,W6-009,"ТОВАРИСТВО З ОБМЕЖЕНОЮ ВІДПОВІДАЛЬНІСТЮ ""ТРАФІ...",39612538,Світлофори / дорожня автоматика,Pilot now,83,I1,4,3,trafficmg.com.ua
9,W6-010,"ТОВ ""ЗАХІД БУДІНЖИНІРИНГ""",44782373,Спортивні покриття,Expansion discovery,79,I1,7,6,


In [5]:
lane_table = pd.Series(lane_counts, name='companies').rename_axis('recommended route').reset_index()
lane_table['share'] = (lane_table['companies'] / len(df)).map(lambda value: f'{value:.1%}')
display(lane_table)

,recommended route,companies,share
0,Expansion discovery,1375,71.3%
1,Pilot now,512,26.5%
2,Requalify,34,1.8%
3,Later / requirements,8,0.4%


In [6]:
segment_counts = wave6_df['segment'].value_counts().rename_axis('Wave 6 segment').reset_index(name='new companies')
display(segment_counts.head(15))

,Wave 6 segment,new companies
0,Гідротехнічні споруди,30
1,Підпірні конструкції,29
2,Резервуари / промислові ємності,28
3,Промислова електрика / ПНР,20
4,Свердловини / водозабір,19
5,Світлофори / дорожня автоматика,18
6,Когенерація,14
7,Спортивні покриття,13
8,Промисловий демонтаж,13
9,Днопоглиблення / порти,10


## Takeaways

1. The lead pool is materially broader than sanitary-technical and electrical installation alone. Wave 6 contributes 241 net-new companies after merging 44 overlaps.
2. The nearest-term motion remains split: 512 accounts can receive an N.14/N.15 pilot message, while 1,375 should receive an honest workflow-discovery message.
3. All prior 51 leads remain present and high confidence.
4. The database is ready for prioritization and outreach, but not for claiming validated demand: the current evidence base still has zero replies, interviews or pilot commitments.